In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

def is_checkout(path):
    return (path / 'train.py').is_file() and (path / 'src' / 'workrunner').is_dir()

cwd = Path.cwd()
ROOT = cwd if is_checkout(cwd) else next(path for path in sorted(cwd.iterdir()) if is_checkout(path))
sys.path.insert(0, str(ROOT / 'src'))
try:
    get_ipython().history_manager.enabled = False
    get_ipython().history_manager.db_log_output = False
except NameError:
    pass
print("Done importing")

In [ ]:
modules = {'numpy': 'numpy', 'torch': 'torch', 'pydantic': 'pydantic', 'loguru': 'loguru', 'cryptography': 'cryptography', 'ipykernel': 'ipykernel', 'Pillow': 'PIL', 'pyarrow': 'pyarrow'}
missing = [line for line in (ROOT / 'requirements.txt').read_text().splitlines() if importlib.util.find_spec(modules[line.split('>=')[0]]) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Ready')


In [ ]:
import logging

if 'trainer' in globals() and hasattr(trainer, 'stop_training'):
    trainer.stop_training()
    del trainer

def run_training(root):
    root = Path(root).resolve()
    from workrunner.paths import set_checkout_directories
    set_checkout_directories(root)
    logging.basicConfig(level=logging.ERROR, format='%(asctime)s %(message)s', stream=sys.stdout, force=True)
    try:
        import workrunner
        if Path(workrunner.__file__).resolve().parent != (root / 'src/workrunner').resolve():
            raise RuntimeError('Restart the kernel before using this source checkout')
        from workrunner.config import load_config, resolve_S3_settings
        from workrunner.network import routed
        from workrunner.S3_training import run_S3_training, validate_S3_endpoint, validate_S3_identity
        from workrunner.vpn import ManagedVPN
        config = load_config(root / 'config.json')
        if config['training']['job_sequence'][0]['max_gpu_utilization'] == -1:
            return 0
        config['S3'] = resolve_S3_settings(config['S3'])
        if config['S3']['protocol'] != 'WR':
            raise ValueError('WR is required')
        if not config['vpn']['enabled'] or not config['vpn']['link']:
            raise ValueError('An enabled VPN link is required')
        validate_S3_endpoint(config['S3'])
        validate_S3_identity(config['S3'], validate_address=False)
        with ManagedVPN(config['vpn']) as proxy, routed(proxy):
            return run_S3_training(config)
    except KeyboardInterrupt:
        print('Training interrupted.', flush=True)
        return 130
    except (ValueError, RuntimeError, ImportError, OSError) as error:
        print(f'Training failed ({type(error).__name__}): {error}', flush=True)
        return 1

exit_code = run_training(ROOT)
